# 03 Urban Indices Exploration

            This notebook computes and inspects NDVI, NDBI, MNDWI, and BSI. These layers explain vegetation,
            built-up intensity, moisture/water signal, and bare-soil exposure.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("Urban Indices Exploration", YEAR)

## Step 1 - Check reflectance band availability

In [ ]:
import pandas as pd

            band_patterns = {
                "blue / SR_B2": [f"*{YEAR}*SR_B2*.tif", f"*{YEAR}*SR_B2*.TIF"],
                "green / SR_B3": [f"*{YEAR}*SR_B3*.tif", f"*{YEAR}*SR_B3*.TIF"],
                "red / SR_B4": [f"*{YEAR}*SR_B4*.tif", f"*{YEAR}*SR_B4*.TIF"],
                "nir / SR_B5": [f"*{YEAR}*SR_B5*.tif", f"*{YEAR}*SR_B5*.TIF"],
                "swir1 / SR_B6": [f"*{YEAR}*SR_B6*.tif", f"*{YEAR}*SR_B6*.TIF"],
            }
            rows = []
            for band, patterns in band_patterns.items():
                matches = find_files("data/raw/landsat", patterns)
                rows.append({"band": band, "count": len(matches), "first_match": str(matches[0].relative_to(PROJECT_ROOT)) if matches else ""})
            pd.DataFrame(rows)

## Step 2 - Review formulas

            - NDVI = `(NIR - RED) / (NIR + RED)`
            - NDBI = `(SWIR1 - NIR) / (SWIR1 + NIR)`
            - MNDWI = `(GREEN - SWIR1) / (GREEN + SWIR1)`
            - BSI = `((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))`

In [ ]:
run_command(["python", "scripts/04_compute_urban_indices.py", "--year", YEAR], dry_run=not RUN_COMMANDS)

## Step 3 - Inspect generated index rasters

In [ ]:
import pandas as pd

            index_paths = {
                "NDVI": project_path(f"data/processed/indices/ndvi_{YEAR}.tif"),
                "NDBI": project_path(f"data/processed/indices/ndbi_{YEAR}.tif"),
                "MNDWI": project_path(f"data/processed/indices/mndwi_{YEAR}.tif"),
                "BSI": project_path(f"data/processed/indices/bsi_{YEAR}.tif"),
            }
            rows = []
            for name, path in index_paths.items():
                if path.exists():
                    rows.append({"index": name, **raster_stats(path)})
                else:
                    rows.append({"index": name, "min": None, "mean": None, "max": None, "valid_pixels": 0})
            pd.DataFrame(rows)

## Step 4 - Plot each index

In [ ]:
cmaps = {"NDVI": "RdYlGn", "NDBI": "YlOrBr", "MNDWI": "Blues", "BSI": "copper"}
            for name, path in index_paths.items():
                plot_raster(path, f"{name} {YEAR}", cmap=cmaps[name])

## Step 5 - Quick interpretation prompts

Use these questions to write your analysis notes after plotting.

In [ ]:
prompts = [
                "Where are low-NDVI surfaces concentrated?",
                "Do high-NDBI zones correspond to high LST zones?",
                "Are water/moisture signals visible in MNDWI?",
                "Does BSI separate bare/peri-urban surfaces from dense built-up areas?",
            ]
            for item in prompts:
                print("-", item)